# 🧹 dfclean — Tutorial interactivo

> Limpieza automática de DataFrames con una API fluent compatible con scikit-learn.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamaraschnaas/dfclean/blob/main/notebooks/tutorial.ipynb)

**¿Qué aprenderás?**
- Instalar `dfclean` desde PyPI
- Construir un pipeline de limpieza paso a paso
- Imputar nulos con distintas estrategias
- Detectar y tratar outliers
- Validar datos con schema declarativo
- Generar un reporte HTML automático

## 1️⃣ Instalación

In [ ]:
!pip install dfclean -q
print('✅ dfclean instalado correctamente')

## 2️⃣ Crear un DataFrame sucio de ejemplo

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 300

df_raw = pd.DataFrame({
    'Age':         np.concatenate([np.random.randint(18, 80, n-5), [999, -5, 150, np.nan, np.nan]]),
    'Salary':      np.concatenate([np.random.normal(50000, 15000, n-3), [1e9, np.nan, np.nan]]),
    'Name':        ['Alice', 'Bob', '  Charlie  ', '', None] * (n//5),
    'Join Date':   ['2020-01-15', '2021-06-30', 'not-a-date', None, '2019-12-01'] * (n//5),
    'Status':      ['active', 'inactive', 'Active', 'INACTIVE', None] * (n//5),
    'constante':   ['mismo_valor'] * n,
    'muchos_nulos':[np.nan] * (n-3) + [1, 2, 3],
})

df_raw = pd.concat([df_raw, df_raw.iloc[:20]], ignore_index=True)

print(f'Shape original: {df_raw.shape}')
print(f'Nulos totales:  {df_raw.isna().sum().sum()}')
print(f'Duplicados:     {df_raw.duplicated().sum()}')
df_raw.head(10)

## 3️⃣ Pipeline completo en una sola expresión

In [ ]:
from dfclean import CleanPipeline

pipeline = (
    CleanPipeline(verbose=True)
    .standardize_columns()                    # Age -> age, Join Date -> join_date
    .replace_empty_strings()                  # '' y '  ' -> NaN
    .drop_duplicates()                        # elimina filas duplicadas
    .drop_constant_columns()                  # elimina columna 'constante'
    .drop_high_null_columns(threshold=0.6)    # elimina muchos_nulos
    .impute_nulls(strategy='smart')           # mediana para num, moda para cat
    .fix_types()                              # detecta fechas y categorias
    .remove_outliers(method='iqr', treatment='clip')  # recorta outliers
    .memory_optimize()                        # downcast RAM
)

df_limpio = pipeline.fit_transform(df_raw)
print(f'\nShape final: {df_limpio.shape}')
df_limpio.head()

## 4️⃣ Reporte de limpieza

In [ ]:
pipeline.report()

## 5️⃣ Detalle por columna

In [ ]:
pd.DataFrame(pipeline.report_.to_dict()['columns'])

## 6️⃣ NullImputer — estrategias por tipo de columna

In [ ]:
from dfclean import NullImputer

imp = NullImputer(
    numeric_strategy='median',
    categorical_strategy='mode',
    null_threshold=60.0
)
df_imp = imp.fit_transform(df_raw.copy())
print('Nulos despues de imputar:', df_imp.isna().sum().sum())
print('Columnas eliminadas:', imp._cols_to_drop)

## 7️⃣ OutlierDetector — resumen de outliers

In [ ]:
from dfclean import OutlierDetector

det = OutlierDetector(method='iqr', treatment='clip', threshold=1.5)
summary = det.outlier_summary(df_raw[['Age','Salary']].dropna())
print(summary)

## 8️⃣ DataFrameSchema — validación declarativa

In [ ]:
from dfclean import ColumnSchema, DataFrameSchema

schema = DataFrameSchema({
    'Age':    ColumnSchema(dtype='float64', min_value=0, max_value=120, nullable=False),
    'Status': ColumnSchema(allowed_values=['active','inactive','Active','INACTIVE']),
})

df_validado = schema.apply(df_raw)
print('Reporte de validacion:')
print(schema.validation_report())

## 9️⃣ Uso estilo scikit-learn: fit en train, transform en test

In [ ]:
train = df_raw.iloc[:250].copy()
test  = df_raw.iloc[250:].copy()

pipe = (
    CleanPipeline()
    .standardize_columns()
    .replace_empty_strings()
    .impute_nulls(strategy='smart')
    .fix_types()
    .remove_outliers(method='iqr', treatment='clip')
)

pipe.fit(train)
train_limpio = pipe.transform(train)
test_limpio  = pipe.transform(test)

print('Train limpio:', train_limpio.shape)
print('Test limpio: ', test_limpio.shape)

## 🔟 Guardar reporte HTML

In [ ]:
path = pipeline.report_.save_html('reporte_dfclean.html')
print(f'Reporte guardado en: {path}')

try:
    from google.colab import files
    files.download('reporte_dfclean.html')
except ImportError:
    print('(Descarga disponible solo en Colab)')

---

## ¡Listo! 🎉

Aprendiste a usar `dfclean` para limpiar DataFrames de forma automática y reproducible.

**Recursos:**
- 📦 [PyPI](https://pypi.org/project/dfclean/)
- 💻 [GitHub](https://github.com/tamaraschnaas/dfclean)
- 🐛 [Reportar un problema](https://github.com/tamaraschnaas/dfclean/issues)